In [0]:
-- # Gold Layer — Aggregated Business Tables with Partitioning & Validation

-- ## Design Decisions
-- - **`%run 0.config`** — zero hardcoded paths or schema names.
-- - **`CREATE OR REPLACE TABLE`** — Gold is a derived/aggregated layer; full replacement is intentional and correct here since aggregates must be recalculated from the full Silver dataset.
-- - **Partitioned by `nationality`** on `driver_wins` — supports filtering by nationality in BI tools.
-- - **Row-count validation** ensures the Gold table is never left empty after a run.

In [0]:
%run ./0.config

In [0]:
--# ── Step 1: Build gold.driver_wins (partitioned, full recalculation) ─────────

spark.sql(f"USE CATALOG {CATALOG_NAME}")

spark.sql(f"""
CREATE OR REPLACE TABLE {fq(GOLD_SCHEMA, 'driver_wins')}
USING DELTA
PARTITIONED BY (nationality)
TBLPROPERTIES ('quality' = 'gold')
AS
SELECT
    d.name,
    d.nationality,
    COUNT(*) AS number_of_wins
FROM  {fq(SILVER_SCHEMA, 'drivers')} d
JOIN  {fq(SILVER_SCHEMA, 'results')} r
  ON  d.driver_id = r.driver_id
WHERE r.position = 1
GROUP BY d.name, d.nationality
ORDER BY number_of_wins DESC
""")

In [0]:
--# ── Step 2: Data Quality Validation ──────────────────────────────────────────

from pyspark.sql import functions as F

gold_df = spark.table(fq(GOLD_SCHEMA, "driver_wins"))
gold_count  = gold_df.count()
null_name   = gold_df.filter(F.col("name").isNull()).count()
null_wins   = gold_df.filter(F.col("number_of_wins") <= 0).count()

assert gold_count > 0,   f"[DQ FAIL] {fq(GOLD_SCHEMA, 'driver_wins')}: table is empty after build!"
assert null_name  == 0,  f"[DQ FAIL] {null_name} NULL driver names in driver_wins"
assert null_wins  == 0,  f"[DQ FAIL] {null_wins} rows with 0 or negative win counts"

print(f"[DQ PASS] {fq(GOLD_SCHEMA, 'driver_wins')}: {gold_count:,} drivers with at least 1 win")
gold_df.orderBy(F.col("number_of_wins").desc()).display()